In [111]:
import pandas as pd

In [112]:
dc = pd.read_parquet("../data/raw/divina_commedia.parquet")
vn1 = pd.read_csv("../data/raw/vita_nuova_poetry.csv")
vn2 = pd.read_csv("../data/raw/vita_nuova_prose.csv")

In [113]:
vn1_column = [
    "chapter_number",
    "poem_index_in_chapter",
    "poem_type",
    "verse_number",
    "text",
]
vn1 = vn1[vn1_column]

vn2_column = ["chapter_number", "paragraph_index", "text"]
vn2 = vn2[vn2_column]

In [114]:
dc["work"] = "divina_commedia"
dc["form"] = "verse"
dc.columns = ["poem_type", "section", "unit", "verse_number", "text", "work", "form"]
dc.head()

,poem_type,section,unit,verse_number,text,work,form
0,Inferno,Canto I,1,1,Nel mezzo del cammin di nostra vita,divina_commedia,verse
1,Inferno,Canto I,1,2,"mi ritrovai per una selva oscura,",divina_commedia,verse
2,Inferno,Canto I,1,3,ché la diritta via era smarrita.,divina_commedia,verse
3,Inferno,Canto I,2,4,Ahi quanto a dir qual era è cosa dura,divina_commedia,verse
4,Inferno,Canto I,2,5,esta selva selvaggia e aspra e forte,divina_commedia,verse


In [115]:
vn1["work"] = "vita_nuova"
vn1["form"] = "verse"
vn1.columns = ["section", "unit", "poem_type", "verse_number", "text", "work", "form"]
vn1.head()

,section,unit,poem_type,verse_number,text,work,form
0,3,1,sonetto,1,"A ciascun'alma presa, e gentil core,",vita_nuova,verse
1,3,1,sonetto,2,"nel cui cospetto ven lo dir presente,",vita_nuova,verse
2,3,1,sonetto,3,in ciò che mi rescrivan suo parvente,vita_nuova,verse
3,3,1,sonetto,4,"salute in lor segnor, cioè Amore.",vita_nuova,verse
4,3,1,sonetto,5,Già eran quasi che atterzate l'ore,vita_nuova,verse


In [132]:
vn2["work"] = "vita_nuova"
vn2["form"] = "prose"
vn2["poem_type"] = "prose"
vn2["verse_number"] = vn2["verse_number"] = pd.NA
vn2.columns = ["verse_number", "section", "unit", "text", "work", "form", "poem_type"]
vn2.head()

,verse_number,section,unit,text,work,form,poem_type
0,vita_nuova,prose,prose,1,1,<NA>,"In quella parte del libro de la mia memoria, d..."
1,vita_nuova,prose,prose,2,1,<NA>,[I] Nove fiate già appresso lo mio nascimento ...
2,vita_nuova,prose,prose,3,1,<NA>,"[II] Poi che furono passati tanti die, che app..."
3,vita_nuova,prose,prose,3,2,<NA>,Questo sonetto si divide in due parti; che la ...
4,vita_nuova,prose,prose,3,3,<NA>,A questo sonetto fue risposto da molti e di di...


In [117]:
column_order = ["work", "form", "poem_type", "section", "unit", "verse_number", "text"]

dc = dc[column_order]
vn1 = vn1[column_order]
vn2 = vn2[column_order]

,work,form,poem_type,section,unit,verse_number,text
0,divina_commedia,verse,Inferno,Canto I,1,1,Nel mezzo del cammin di nostra vita
1,divina_commedia,verse,Inferno,Canto I,1,2,"mi ritrovai per una selva oscura,"
2,divina_commedia,verse,Inferno,Canto I,1,3,ché la diritta via era smarrita.
3,divina_commedia,verse,Inferno,Canto I,2,4,Ahi quanto a dir qual era è cosa dura
4,divina_commedia,verse,Inferno,Canto I,2,5,esta selva selvaggia e aspra e forte


### eda

In [127]:
# transform roman numbers in arabic numbers
ROMAN_VALUES = {"I": 1, "V": 5, "X": 10, "L": 50, "C": 100}


def roman_to_int(roman: str) -> int:
    """Convert a roman numeral to an integer"""
    total = 0
    for i, letter in enumerate(roman):
        value = ROMAN_VALUES[letter]
        if i + 1 < len(roman) and value < ROMAN_VALUES[roman[i + 1]]:
            total -= value
        else:
            total += value
    return total

In [128]:
dc["section"] = dc["section"].str.removeprefix("Canto ").map(roman_to_int)

In [131]:
df = pd.concat([dc, vn1, vn2], ignore_index=True)
df["verse_number"] = df["verse_number"].astype("Int64")
df.head()

,work,form,poem_type,section,unit,verse_number,text
0,divina_commedia,verse,Inferno,1,1,1,Nel mezzo del cammin di nostra vita
1,divina_commedia,verse,Inferno,1,1,2,"mi ritrovai per una selva oscura,"
2,divina_commedia,verse,Inferno,1,1,3,ché la diritta via era smarrita.
3,divina_commedia,verse,Inferno,1,2,4,Ahi quanto a dir qual era è cosa dura
4,divina_commedia,verse,Inferno,1,2,5,esta selva selvaggia e aspra e forte


In [134]:
import re


def clean_text(text: str) -> str:
    """Remove markup and normalise punctuation in a verse or paragraph."""
    text = text.replace("}}", "")  # leftover wiki markup
    text = re.sub(r"<[^>]+>", "", text)  # HTML tags like </span>
    text = re.sub(r"\[[IVXLC]+\]\s*", "", text)  # chapter markers like [II]
    text = text.replace("’", "'")  # one apostrophe type
    text = text.replace("«", '"').replace("»", '"')  # one quote type
    return text.strip()

In [135]:
df["text"] = df["text"].map(clean_text)